In [1]:
import torch
import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader
import torch.nn as nn

import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset

import matplotlib.pyplot as plt
import seaborn as sns

import os
import numpy as np
import pandas as pd
import cv2

import pathlib
from PIL import Image

In [2]:
# Define variables for model from scratch
NUM_CLASSES = 2
NUM_CHANNELS = 1
BATCH_SIZE = 64
IMG_SIZE = 224
PATCH_SIZE = 16
NUM_PATCHES = (IMG_SIZE // PATCH_SIZE)**2
EMBEDDING_DIM = 128
ATTENTION_HEADS = 4
TRANSFORMER_BLOCKS = 4
MLP_HIDDEN_NODES = 256
LEARNING_RATE = 3e-4
EPOCHS = 50

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

main_path = "/kaggle/input/datasets/uyendang2404/nih-chest-balanced-split-v5"

In [3]:
class PatchEmbedding(torch.nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.patch_embed = nn.Conv2d(in_channels=NUM_CHANNELS,
                                     out_channels=EMBEDDING_DIM,
                                     kernel_size=PATCH_SIZE,
                                     stride=PATCH_SIZE)
    
    def forward(self, x):
        # Patch_embedding
        x = self.patch_embed(x)
        # Flattening
        x = x.flatten(2)
        # Position Embedding
        x = x.transpose(1,2)
    
        return x

class TransformerEncoder(nn.Module):
    def __init__(self):
        super().__init__()

        self.layer_norm1 = nn.LayerNorm(EMBEDDING_DIM)
        self.layer_norm2 = nn.LayerNorm(EMBEDDING_DIM)

        # Add dropout inside attention
        self.multihead_attention = nn.MultiheadAttention(
            EMBEDDING_DIM,
            ATTENTION_HEADS,
            dropout=0.1,
            batch_first=True
        )

        # Dropout layers
        self.dropout1 = nn.Dropout(0.1)
        self.dropout2 = nn.Dropout(0.1)

        # Improved MLP block
        self.mlp = nn.Sequential(
            nn.Linear(EMBEDDING_DIM, MLP_HIDDEN_NODES),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(MLP_HIDDEN_NODES, EMBEDDING_DIM),
            nn.Dropout(0.1)
        )

    def forward(self, x):

        # Residual 1 (Attention)
        residual = x
        x = self.layer_norm1(x)
        x = self.multihead_attention(x, x, x)[0]
        x = self.dropout1(x)
        x = x + residual

        # Residual 2 (MLP)
        residual = x
        x = self.layer_norm2(x)
        x = self.mlp(x)
        x = x + residual

        return x


class MLP_Head(torch.nn.Module):
  def __init__(self) -> None:
    super().__init__()
    self.layer_norm = nn.LayerNorm(EMBEDDING_DIM)
    self.dropout = nn.Dropout(0.1)
    self.mlp_head = nn.Linear(EMBEDDING_DIM, NUM_CLASSES)

  def forward(self, x):
    x = self.layer_norm(x)
    x = self.dropout(x)
    x = self.mlp_head(x)

    return x


class VisionTransformer(torch.nn.Module):
  def __init__(self) -> None:
    super().__init__()
    self.patch_embedding = PatchEmbedding()
    # Class token
    self.cls_token = nn.Parameter(torch.randn(1, 1, EMBEDDING_DIM))
    # Position Embedding
    self.postion_embedding = nn.Parameter(torch.randn(1, NUM_PATCHES + 1, EMBEDDING_DIM))
    # Dropout
    self.dropout = nn.Dropout(0.1)

    self.transformer_blocks = nn.Sequential(
        *[TransformerEncoder() for _ in range(TRANSFORMER_BLOCKS)]
        )

    self.mlp_head = MLP_Head()

  def forward(self, x):
    x = self.patch_embedding(x)
    class_token = self.cls_token.expand(x.size(0), -1, -1)
    x = torch.cat([class_token, x], dim=1)
    x = x + self.postion_embedding
    x = self.dropout(x)
    x = self.transformer_blocks(x)
    x = x[:, 0]
    x = self.mlp_head(x)

    return x

In [4]:
class ChestXrayBiasDataset(Dataset):
    def __init__(self, dataframe, image_dir, transform=None):
        """
        Args:
            dataframe: The filtered dataframe (train_df, val_df, or test_df)
            image_dir: Path to the specific split folder (e.g., 'nih_split_data/train')
            transform: PyTorch transforms
        """
        self.df = dataframe.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # Mapping 'Target' column to the actual subfolder names
        subfolder = row['Target'] 
        img_path = os.path.join(self.image_dir, subfolder, row['Image Index'])

        # Load image as Grayscale 'L'
        image = Image.open(img_path).convert('L') 

        # Create Binary Label: 1 for Disease, 0 for No Finding
        binary_label = 1.0 if row['Target'] == 'Disease' else 0.0
        label = torch.tensor(binary_label).float()
        
        # Bias features (Protected attributes)
        gender = 1.0 if row['Patient Gender'] == 'F' else 0.0
        position = 1.0 if row['View Position'] == 'AP' else 0.0

        if self.transform:
            image = self.transform(image)

        return image, label, gender, position


# Improved CSV loading and filtering
def create_split_df(base_path, sub_path, split_name):
    # Load csv file
    csv_path = os.path.join(base_path, 'balanced_subsample.csv')
    master_df = pd.read_csv(csv_path)
    
    # Filter by the split (train, val, or test)
    split_df = master_df[master_df['Split'] == split_name].copy()
    
    # Check the specific subfolder for this split to verify files exist
    split_folder_path = os.path.join(base_path, sub_path, split_name)
    
    physical_files = []
    for subfolder in ['Disease', 'No Finding']:
        path = os.path.join(split_folder_path, subfolder)
        if os.path.exists(path):
            physical_files.extend(os.listdir(path))
            
    # Keep only rows where the image actually exists
    final_df = split_df[split_df['Image Index'].isin(physical_files)]
    return final_df.reset_index(drop=True)

# --- Enhancement function ---
def medical_enhancement_pil(img):
    img_np = np.array(img)

    # Ensure grayscale FIRST (important)
    if len(img_np.shape) == 3:
        img_np = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)

    # Percentile clipping (after grayscale)
    lower, upper = np.percentile(img_np, (1, 99))
    img_np = np.clip(img_np, lower, upper)

    # Normalize to uint8
    img_np = cv2.normalize(img_np, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

    # CLAHE
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(img_np)

    return Image.fromarray(enhanced)


# --- Transforms ---
# data_transform = transforms.Compose([
#         transforms.Resize((224, 224)),
#         #transforms.Grayscale(num_output_channels=1),
#         transforms.Lambda(medical_enhancement_pil),
#         transforms.ToTensor(),
#         transforms.Normalize(
#             mean=[0.5],  # single channel
#             std=[0.5]),
#     ])
data_transform = transforms.Compose([
                        transforms.Resize((IMG_SIZE, IMG_SIZE)),
                        transforms.Grayscale(num_output_channels=1),
                        transforms.Lambda(medical_enhancement_pil),
                        transforms.ToTensor(),
                        transforms.Normalize(
                            mean=[0.5],  # single channel
                            std=[0.5]),
                    ])
csv_path = main_path + 'balanced_subsample.csv'

# Data Loading Setup
train_df = create_split_df(main_path,'sub_data', 'train')
val_df = create_split_df(main_path,'sub_data', 'val')
test_df  = create_split_df(main_path, 'sub_data', 'test')

train_dataset = ChestXrayBiasDataset(train_df, os.path.join(main_path, 'sub_data/train'), data_transform)
val_dataset = ChestXrayBiasDataset(val_df, os.path.join(main_path, 'sub_data/val'), data_transform)
test_dataset  = ChestXrayBiasDataset(test_df,  os.path.join(main_path, 'sub_data/test'),  data_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader  = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE)

In [5]:
from timm import create_model

def getmodel(model_type="scratch"):

    if model_type == "scratch":
        model = VisionTransformer().to(device)
        # optimizer = torch.optim.AdamW(
        #     model.parameters(),
        #     lr=LEARNING_RATE,
        #     weight_decay=1e-4
        # )
        # criterion = nn.CrossEntropyLoss()

    elif model_type == "pretrained":
        pretrained_weights = torchvision.models.ViT_B_16_Weights.DEFAULT
        model = torchvision.models.vit_b_16(weights=pretrained_weights).to(device)

        # Change input channels from 3 -> 1
        old_conv = model.conv_proj
        model.conv_proj = nn.Conv2d(
            in_channels=1,
            out_channels=old_conv.out_channels,
            kernel_size=old_conv.kernel_size,
            stride=old_conv.stride,
            padding=old_conv.padding,
            bias=False
        )

        # Initialize grayscale conv using RGB pretrained weights
        model.conv_proj.weight.data = old_conv.weight.data.mean(dim=1, keepdim=True)

        # Freeze pretrained backbone
        for param in model.parameters():
            param.requires_grad = False
        
        model.heads = nn.Linear(in_features=768, out_features=2).to(device)

    return model.to(device)

In [6]:
def train(model_name,criterion,train_loader,val_loader,epochs=10,lr=3e-4,
          checkpoint_path="last_checkpoint.pth",
          best_loss_checkpoint="best_loss_checkpoint.pth",
          best_acc_checkpoint="best_acc_checkpoint.pth",
          device="cuda"):


    model = getmodel(model_name)
    
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=1e-4
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=epochs
    )
    
    start_epoch = 0
    best_acc = 0.0
    best_val_loss = float("inf")

    if os.path.exists(checkpoint_path):
    
        checkpoint = torch.load(checkpoint_path)
    
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    
        if 'scheduler_state_dict' in checkpoint:
            scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    
        start_epoch = checkpoint['epoch'] + 1
    
        print(f"✅ Resumed training from epoch {start_epoch}")

    if os.path.exists(best_loss_checkpoint):

        best_checkpoint = torch.load(
            best_loss_checkpoint,
            map_location=device
        )

        best_val_loss = best_checkpoint['val_loss']

        print(f"✅ Previous best val loss: {best_val_loss:.4f}")

    if os.path.exists(best_acc_checkpoint):

        best_checkpoint2 = torch.load(
            best_acc_checkpoint,
            map_location=device
        )

        best_acc = best_checkpoint2.get(
            'accuracy',
            0.0
        )

        print(f"✅ Previous best val acc: {best_acc:.2f}%")

    for epoch in range(start_epoch, epochs):
        model.train()
    
        train_loss = 0.0
        correct = 0
        total = 0
    
        for batch_idx, (images, labels, *_) in enumerate(train_loader):
            images = images.to(device)
            labels = labels.to(device).view(-1).long()
    
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
    
            loss.backward()
            optimizer.step()
    
            train_loss += loss.item() * images.size(0)
    
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    
            if batch_idx % 100 == 0:
                batch_acc = 100 * (preds == labels).sum().item() / labels.size(0)
                print(f"Batch {batch_idx+1:3d} | Loss: {loss.item():.4f} | Acc: {batch_acc:.2f}%")
    
        epoch_loss = train_loss / total
        epoch_acc = 100 * correct / total
    
        # ---------------------
        # VALIDATION
        # ---------------------
        model.eval()
    
        val_loss = 0.0
        val_correct = 0
        val_total = 0
    
        with torch.no_grad():
            for val_images, val_labels, * _ in val_loader:
                val_images = val_images.to(device)
                val_labels = val_labels.to(device).view(-1).long()
    
                outputs = model(val_images)
                loss = criterion(outputs, val_labels)
    
                val_loss += loss.item() * val_images.size(0)
    
                preds = outputs.argmax(dim=1)
                val_correct += (preds == val_labels).sum().item()
                val_total += val_labels.size(0)
    
        val_loss = val_loss / val_total
        val_acc = 100 * val_correct / val_total
    
        # ---------------------
        # LOGGING
        # ---------------------
        print(
            f"==> Epoch {epoch+1}: "
            f"Train Loss={epoch_loss:.4f}, Train Acc={epoch_acc:.2f}% | "
            f"Val Loss={val_loss:.4f}, Val Acc={val_acc:.2f}%"
        )
    
        # ---------------------
        # SAVE LAST CHECKPOINT
        # ---------------------
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': epoch_loss,
            'best_acc': best_acc,
            'best_val_loss': best_val_loss
        }, f"last_checkpoint_{model_name}.pth")
    
        # ---------------------
        # SAVE BEST MODEL (by VAL ACC)
        # ---------------------
        if val_acc > best_acc:
            best_acc = val_acc
    
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'accuracy': val_acc,
            }, f"best_acc_{model_name}.pth")
    
            print(f"⭐ Best ACC model saved (Val Acc: {val_acc:.2f}%)")
    
        # ---------------------
        # OPTIONAL: BEST MODEL (by VAL LOSS)
        # ---------------------
        if val_loss < best_val_loss:
            best_val_loss = val_loss
    
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_loss': val_loss,
            }, f"best_loss_{model_name}.pth")
    
            print(f"⭐ Best LOSS model saved (Val Loss: {val_loss:.4f})")

In [7]:
model_name="scratch"
last_checkpoint = "/kaggle/input/datasets/uyendang2404/last-checkpoint-scratch/last_checkpoint_scratch.pth"
best_loss_cp = "/kaggle/input/datasets/uyendang2404/last-checkpoint-scratch/best_loss_scratch.pth"
best_acc_cp = "/kaggle/input/datasets/uyendang2404/last-checkpoint-scratch/best_acc_scratch.pth"
criterion = nn.CrossEntropyLoss()
train(model_name,criterion,train_loader,val_loader,epochs=70,lr=1e-4,
          checkpoint_path=last_checkpoint,
          best_loss_checkpoint=best_loss_cp,
          best_acc_checkpoint=best_acc_cp,
          device=device)


✅ Resumed training from epoch 46
✅ Previous best val loss: 0.6460
✅ Previous best val acc: 63.53%
Batch   1 | Loss: 0.5015 | Acc: 76.56%
Batch 101 | Loss: 0.6503 | Acc: 64.06%
Batch 201 | Loss: 0.5631 | Acc: 68.75%
Batch 301 | Loss: 0.5756 | Acc: 68.75%
Batch 401 | Loss: 0.4860 | Acc: 75.00%
Batch 501 | Loss: 0.5902 | Acc: 67.19%
Batch 601 | Loss: 0.4462 | Acc: 81.25%
==> Epoch 47: Train Loss=0.5571, Train Acc=71.13% | Val Loss=0.7083, Val Acc=60.91%
Batch   1 | Loss: 0.5635 | Acc: 75.00%
Batch 101 | Loss: 0.5423 | Acc: 67.19%
Batch 201 | Loss: 0.5606 | Acc: 78.12%
Batch 301 | Loss: 0.5163 | Acc: 68.75%
Batch 401 | Loss: 0.6502 | Acc: 64.06%
Batch 501 | Loss: 0.6708 | Acc: 62.50%
Batch 601 | Loss: 0.4756 | Acc: 76.56%
==> Epoch 48: Train Loss=0.5541, Train Acc=71.32% | Val Loss=0.7112, Val Acc=61.21%
Batch   1 | Loss: 0.5444 | Acc: 65.62%
Batch 101 | Loss: 0.6159 | Acc: 65.62%
Batch 201 | Loss: 0.6301 | Acc: 59.38%
Batch 301 | Loss: 0.4975 | Acc: 76.56%
Batch 401 | Loss: 0.6153 | Acc: 

In [ ]:
model_name="pretrained"
last_checkpoint = "/kaggle/input/datasets/uyendang2404/last-checkpoint-pretrained/last_checkpoint_pretrained.pth"
best_loss_cp = "/kaggle/input/datasets/uyendang2404/last-checkpoint-pretrained/best_loss_pretrained.pth"
best_acc_cp = "/kaggle/input/datasets/uyendang2404/last-checkpoint-pretrained/best_acc_pretrained.pth"
criterion = nn.CrossEntropyLoss()
train(model_name,criterion,train_loader,val_loader,epochs=20,lr=1e-4,
          checkpoint_path=last_checkpoint,
          best_loss_checkpoint=best_loss_cp,
          best_acc_checkpoint=best_acc_cp,
          device=device)


In [9]:
def test(model_name, test_loader, best_checkpoint, device):

    model = getmodel(model_name)
    checkpoint = torch.load(
        best_checkpoint,
        map_location=device
    )
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()

    data = []
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels, genders, positions in test_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)

            # 🔥 Convert logits → probabilities
            probs = nn.functional.softmax(outputs, dim=1)[:, 1]  # prob of class 1

            preds = outputs.argmax(dim=1)

            total += labels.size(0)
            correct += (preds == labels).sum().item()

            lbls_np = labels.cpu().numpy().flatten()
            preds_np = preds.cpu().numpy().flatten()
            probs_np = probs.cpu().numpy().flatten()

            for i in range(len(preds)):
                data.append({
                    'Gender': 'Female' if genders[i] == 1 else 'Male',
                    'View': 'AP' if positions[i] == 1 else 'PA',
                    'True_Label': int(lbls_np[i]),
                    'Pred_Label': int(preds_np[i]),
                    'Pred_Prob': float(probs_np[i]),  # 🔥 NEW
                    'Correct': 1 if preds_np[i] == lbls_np[i] else 0
                })

    test_acc = 100 * correct / total
    print(f"Test Accuracy: {test_acc:.2f}%")

    df = pd.DataFrame(data)
    df.to_csv(f'/kaggle/working/test_results_{model_name}.csv', index=False)

    return df

In [ ]:
best_checkpoint = "/kaggle/input/datasets/uyendangrs/vit-checkpoint-1/best_acc_scratch_1.pth"
model_name="scratch"
res_df = test(model_name, test_loader, best_checkpoint, device)

In [ ]:
best_checkpoint = "/kaggle/input/datasets/uyendang2404/best-checkpoint-pretrained/best_acc_pretrained (1).pth"
model_name="pretrained"
res_df = test(model_name, test_loader, best_checkpoint, device)